In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings

warnings.filterwarnings('ignore')


df_cons = pd.read_excel('CreditConsumptionData.xlsx')
df_beh = pd.read_excel('CustomerBehaviorData.xlsx')
df_demo = pd.read_excel('CustomerDemographics.xlsx')

# --- 2. Data Merging ---
# Merging all datasets on 'ID'
df = pd.merge(df_cons, df_beh, on='ID', how='inner')
df = pd.merge(df, df_demo, on='ID', how='inner')

# Separating the rows where we need to predict the target (cc_cons is NaN)
final_target_test = df[df['cc_cons'].isnull()]

# Using rows where target is available for training/testing
train_data = df[df['cc_cons'].notnull()]

# Defining Features (X) and Target (y)
# Dropping ID as it is a unique identifier and cc_cons as it is the target
X = train_data.drop(columns=['cc_cons', 'ID'])
y = train_data['cc_cons'] # Keeping y as a Series for model compatibility

# --- 3. Preprocessing Pipeline ---
# Identifying column types
numerical_cols = X.select_dtypes(include=['number', 'float']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Creating a transformer that handles missing values AND scaling/encoding
# Added SimpleImputer because StandardScaler fails on NaN values
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# --- 4. Feature Transformation ---
X_processed_array = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

X_processed = pd.DataFrame(X_processed_array, columns=feature_names)



# --- 5. Model Training ---
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Set up Random Forest with GridSearchCV
rf = RandomForestRegressor(random_state=42)

# Increasing n_estimators for better accuracy
param_grid = {
    'n_estimators': [50, 100], 
    'max_depth': [5, 10],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

# --- 6. Model Evaluation ---
predictions = best_model.predict(X_test)

# Calculating RMSE
rmse = np.sqrt(mean_squared_error(y_test, predictions))

# Calculating RMSPE (Business Requirement)
# Formula: Sqrt of Average of Squared Percentage Errors
rmspe = np.sqrt(np.mean(np.square((y_test - predictions) / y_test))) * 100

print(f"Model Best Params: {grid_search.best_params_}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test RMSPE: {rmspe:.2f}%")

# --- 7. Final Predictions for Missing Values ---
# Preprocessing the data where cc_cons was missing
X_final_test = final_target_test.drop(columns=['cc_cons', 'ID'])
X_final_processed = preprocessor.transform(X_final_test) # Use transform, not fit_transform

final_predictions = best_model.predict(X_final_processed)

# Adding predictions back to the dataframe
final_target_test['predicted_cc_cons'] = final_predictions
print("\nFirst 5 Predicted Values for missing targets:")
print(final_target_test[['ID', 'predicted_cc_cons']].head())